In [ ]:
import pandas as pd
import numpy as np
import re
import cobra
from cobra import Model, Reaction, Metabolite
from cobra.io import load_yaml_model
from cobra.io import load_matlab_model
from cobra.io import load_json_model
from cobra.io import read_sbml_model
from itertools import combinations
import memote
import memote.support.consistency

In [ ]:
def load_metabolic_model(modelfile, model_id):
        
    file_extension = model_id.lower().split('.')[-1]
    
    if file_extension == 'xml' or file_extension == 'sbml':
        try:
            model = read_sbml_model(modelfile+model_id)
            return model
        except:
            print("Something error with the model!")
    elif file_extension == 'json':
        try:
            model = load_json_model(modelfile+model_id)
            return model
        except:
            print("Something error with the model!")
    elif file_extension == 'yml':
        try:
            model = load_yaml_model(modelfile+model_id)
            return model
        except:
            print("Something error with the model!")
    elif file_extension == 'mat':
        try:
            model = load_matlab_model(modelfile+model_id)
            return model
        except:
            print("Something error with the model!")
    else:
        return "Unsupported file format!"
    

def find_dead_end_mets(model):
    # Using the 'find_blocked_metabolites' from memote to find dead-end metabolites
    # "1" means Dead-end metabolites and "-1" means Orphan metabolites. 
    # "Processes" means how many cores would you like to use for calculating.
    Orphan_met = memote.support.consistency.find_blocked_metabolites(model, -1, processes=8)
    Deadend_met = memote.support.consistency.find_blocked_metabolites(model, 1, processes=8)
    all_blocked_mets = Orphan_met + Deadend_met
    unique_all_mets = list(set(all_blocked_mets))

    # print(len(unique_all_mets))
    return unique_all_mets

# load model
model = load_metabolic_model('../Models/', 'Human2.mat')
blocked_rxns = cobra.flux_analysis.find_blocked_reactions(model) # Blocked reactions list
Deadend_mets = find_dead_end_mets(model) # Deadend metabolites list

print(len(blocked_rxns))
print(len(Deadend_mets))